In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE, ADASYN, KMeansSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss, ClusterCentroids
from imblearn.combine import SMOTEENN

# Load the dataset
df = pd.read_csv(r"T:\00 445\student\student-mat.csv", sep=";")

# Create binary target variable: 1 = Pass (G3 ≥ 10), 0 = Fail
df['pass'] = df['G3'].apply(lambda x: 1 if x >= 10 else 0)


#df.drop(columns=['G2','Medu','Fedu','studytime','famrel', 'G3'], inplace=True)
#df.drop(columns=['G2','G1', 'G3'], inplace=True)
#df.drop(columns=['G1','G2','Medu','Fedu','studytime','famrel', 'G3'], inplace=True)
df.drop(columns=['G2','G3'], inplace=True)
#df.drop(columns=['G3'], inplace=True)

# Separate features and target
X = df.drop(columns=['pass'])
y = df['pass']

# One-Hot Encoding for categorical columns
categorical_cols = X.select_dtypes(include='object').columns.tolist()
column_transformer = ColumnTransformer([("onehot", OneHotEncoder(drop='first'), categorical_cols)], remainder='passthrough')

# Transform the dataset
X_encoded = column_transformer.fit_transform(X)

# Convert the transformed data back to DataFrame
X_df = pd.DataFrame(X_encoded.toarray() if hasattr(X_encoded, 'toarray') else X_encoded)

# Split the dataset: 80% train, 5% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.55, stratify=y_temp, random_state=42)


# Flag to choose the sampling technique
SAMPLING_TECHNIQUE = 'ADASYN'
  # Options: 'SMOTE', 'ADASYN', 'SMOTEENN', 'UnderSampling', 'TomekLinks', 'NearMiss', 'ClusterCentroids', 'KMeansSMOTE'

# Apply selected sampling technique
if SAMPLING_TECHNIQUE == 'SMOTE':
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print("After SMOTE:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'ADASYN':
    adasyn = ADASYN(random_state=42)
    X_train_bal, y_train_bal = adasyn.fit_resample(X_train, y_train)
    print("After ADASYN:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'SMOTEENN':
    smote_enn = SMOTEENN(random_state=42)
    X_train_bal, y_train_bal = smote_enn.fit_resample(X_train, y_train)
    print("After SMOTEENN:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'UnderSampling':
    undersampler = RandomUnderSampler(random_state=42)
    X_train_bal, y_train_bal = undersampler.fit_resample(X_train, y_train)
    print("After Under-sampling:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'TomekLinks':
    tomek = TomekLinks()
    X_train_bal, y_train_bal = tomek.fit_resample(X_train, y_train)
    print("After Tomek Links:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'NearMiss':
    near_miss = NearMiss(version=1)  # Use version 1, 2, or 3
    X_train_bal, y_train_bal = near_miss.fit_resample(X_train, y_train)
    print("After NearMiss:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'ClusterCentroids':
    cluster_centroids = ClusterCentroids(random_state=42)
    X_train_bal, y_train_bal = cluster_centroids.fit_resample(X_train, y_train)
    print("After Cluster Centroids:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'KMeansSMOTE':
    kmeans_smote = KMeansSMOTE(random_state=42)
    X_train_bal, y_train_bal = kmeans_smote.fit_resample(X_train, y_train)
    print("After KMeansSMOTE:")
    print(pd.Series(y_train_bal).value_counts())
else:
    # If no sampling technique is selected, use the original training data
    X_train_bal, y_train_bal = X_train, y_train

# Standardize features for training, validation, and test sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# You can now proceed with model training (e.g., Decision Tree, Random Forest, etc.)


After ADASYN:
pass
0    199
1    185
Name: count, dtype: int64


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
# Step 1: Select only numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Calculate the correlation matrix for all numerical columns
correlation_matrix_all = df[numerical_cols].corr()

# Step 2: Plot the correlation matrix as a heatmap
plt.figure(figsize=(12, 10))  # Increase figure size for better readability
sns.heatmap(correlation_matrix_all, annot=True, cmap='coolwarm', fmt='.2f', cbar=True, annot_kws={'size': 12, 'weight': 'bold'}, 
            linewidths=0.5, linecolor='gray', vmin=-1, vmax=1)  # Added grid lines and color scale range
plt.title("Correlation Matrix of All Features", fontsize=16, weight='bold')
plt.xticks(rotation=45, ha='right', fontsize=12)  # Rotate x-axis labels for better visibility
plt.yticks(rotation=0, ha='right', fontsize=12)  # Keep y-axis labels horizontal for clarity
plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

# Step 3: Print the full correlation matrix
print("Correlation matrix of all features:")
print(correlation_matrix_all)


NameError: name 'sns' is not defined

<Figure size 1200x1000 with 0 Axes>

In [ ]:


# Initialize Decision Tree Classifier with default settings
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model on the balanced, scaled training data
dt_model.fit(X_train_scaled, y_train_bal)

# Predictions on validation and test sets
y_val_pred = dt_model.predict(X_val_scaled)
y_test_pred = dt_model.predict(X_test_scaled)

# Calculate classification metrics
val_report = classification_report(y_val, y_val_pred, output_dict=True)
test_report = classification_report(y_test, y_test_pred, output_dict=True)

# Accuracy comparison
val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Confusion matrices
val_conf_matrix = confusion_matrix(y_val, y_val_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

# Display classification reports as text
print("Classification Report (Validation Set):")
print(classification_report(y_val, y_val_pred))

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

# Display accuracy for validation and test sets
print("\nAccuracy Comparison:")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Plotting accuracy comparison between validation and test sets
fig, ax = plt.subplots()
ax.bar(['Validation', 'Test'], [val_accuracy, test_accuracy], color=['blue', 'green'])
ax.set_title('Accuracy Comparison between Validation and Test')
ax.set_ylabel('Accuracy')
plt.show()

# Plotting other metrics comparison: precision, recall, and F1-score
metrics = ['precision', 'recall', 'f1-score']
metrics_values = {metric: [val_report['1'][metric], test_report['1'][metric]] for metric in metrics}

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(metrics):
    ax[i].bar(['Validation', 'Test'], metrics_values[metric], color=['blue', 'green'])
    ax[i].set_title(f'{metric.capitalize()} Comparison')
    ax[i].set_ylabel(metric.capitalize())
plt.tight_layout()
plt.show()

# Plotting confusion matrix for better comparison with annotations
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# For Validation Confusion Matrix
ax[0].imshow(val_conf_matrix, cmap='Blues', interpolation='nearest')
ax[0].set_title('Validation Confusion Matrix')
for i in range(val_conf_matrix.shape[0]):
    for j in range(val_conf_matrix.shape[1]):
        ax[0].text(j, i, str(val_conf_matrix[i, j]), ha='center', va='center', color='black')
ax[0].set_xticks([0, 1])
ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(['Predicted Fail', 'Predicted Pass'])
ax[0].set_yticklabels(['Actual Fail', 'Actual Pass'])

# For Test Confusion Matrix
ax[1].imshow(test_conf_matrix, cmap='Blues', interpolation='nearest')
ax[1].set_title('Test Confusion Matrix')
for i in range(test_conf_matrix.shape[0]):
    for j in range(test_conf_matrix.shape[1]):
        ax[1].text(j, i, str(test_conf_matrix[i, j]), ha='center', va='center', color='black')
ax[1].set_xticks([0, 1])
ax[1].set_yticks([0, 1])
ax[1].set_xticklabels(['Predicted Fail', 'Predicted Pass'])
ax[1].set_yticklabels(['Actual Fail', 'Actual Pass'])

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import pandas as pd

# Initialize DecisionTreeClassifier with class_weight='balanced' for class imbalance handling
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Updated parameter grid to align with previous suggestions
param_grid_dt = {
    "criterion": ["gini", "entropy"],  # Removed 'log_loss' as it's not commonly used in Decision Trees
    "max_depth": [3, 5, 10, 15, None],  # Avoid overfitting by restricting tree depth, but leave None for no limit
    "min_samples_split": [2, 5, 10],  # Adjust based on your data size
    "min_samples_leaf": [1, 2, 4, 8],  # Try different leaf sizes to avoid overfitting
    "class_weight": ['balanced', None]  # Use 'balanced' to adjust for the class imbalance, or leave it as None
}

# Initialize GridSearchCV with 5-fold cross-validation and recall as scoring metric
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid_dt, cv=5, n_jobs=-1, scoring='recall')

# Fit GridSearchCV to the training data
grid_search.fit(X_train_scaled, y_train_bal)

# Best parameters from grid search
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

# Best model from grid search
best_dt_model = grid_search.best_estimator_

# Plotting grid search results: Hyperparameter tuning performance visualization for max_depth
results = pd.DataFrame(grid_search.cv_results_)
plt.figure(figsize=(10, 6))
plt.plot(results['param_max_depth'], results['mean_test_score'], label='mean_test_score', marker='o')
plt.title("GridSearchCV Hyperparameter Tuning Performance (Max Depth)")
plt.xlabel('max_depth')
plt.ylabel('Mean Test Score (Recall)')
plt.legend()
plt.show()

# Plot for min_samples_split
plt.figure(figsize=(10, 6))
for min_samples_split in results['param_min_samples_split'].unique():
    subset = results[results['param_min_samples_split'] == min_samples_split]
    plt.plot(subset['param_max_depth'], subset['mean_test_score'], label=f'min_samples_split={min_samples_split}', marker='o')
plt.title('GridSearchCV Performance with respect to Max Depth for Different min_samples_split')
plt.xlabel('Max Depth')
plt.ylabel('Mean Test Score (Recall)')
plt.legend(title='min_samples_split')
plt.show()

# Plot for min_samples_leaf
plt.figure(figsize=(10, 6))
for min_samples_leaf in results['param_min_samples_leaf'].unique():
    subset = results[results['param_min_samples_leaf'] == min_samples_leaf]
    plt.plot(subset['param_max_depth'], subset['mean_test_score'], label=f'min_samples_leaf={min_samples_leaf}', marker='o')
plt.title('GridSearchCV Performance with respect to Max Depth for Different min_samples_leaf')
plt.xlabel('Max Depth')
plt.ylabel('Mean Test Score (Recall)')
plt.legend(title='min_samples_leaf')
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

# Best Parameters for Decision Tree
best_dt_params = {
    'class_weight': None, 
    'criterion': 'entropy', 
    'max_depth': 13, 
    'min_samples_leaf': 1, 
    'min_samples_split': 3
}

# Initialize Decision Tree Classifier with the best hyperparameters
best_dt_model = DecisionTreeClassifier(**best_dt_params, random_state=42)

# Train the model on the balanced, scaled training data
best_dt_model.fit(X_train_scaled, y_train_bal)

# Predictions on validation and test sets with the best model
y_val_pred_dt_best = best_dt_model.predict(X_val_scaled)
y_test_pred_dt_best = best_dt_model.predict(X_test_scaled)

# Classification metrics for the tuned model
val_report_dt_best = classification_report(y_val, y_val_pred_dt_best, output_dict=True)
test_report_dt_best = classification_report(y_test, y_test_pred_dt_best, output_dict=True)

# Accuracy comparison after hyperparameter tuning
val_accuracy_dt_best = accuracy_score(y_val, y_val_pred_dt_best)
test_accuracy_dt_best = accuracy_score(y_test, y_test_pred_dt_best)

# Print the classification report for validation set in the requested format
print("Classification Report (Validation Set):")
print(classification_report(y_val, y_val_pred_dt_best))

# Print the classification report for test set in the requested format
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred_dt_best))

# Display accuracy for validation and test sets
print("\nAccuracy Comparison:")
print(f"Validation Accuracy: {val_accuracy_dt_best:.4f}")
print(f"Test Accuracy: {test_accuracy_dt_best:.4f}")

# Plotting accuracy comparison after hyperparameter tuning
fig, ax = plt.subplots()
ax.bar(['Validation', 'Test'], [val_accuracy_dt_best, test_accuracy_dt_best], color=['blue', 'green'])
ax.set_title('Accuracy Comparison after Hyperparameter Tuning (Decision Tree)')
ax.set_ylabel('Accuracy')
plt.show()

# Plotting other metrics comparison after tuning: precision, recall, and F1-score
metrics_values_dt_best = {metric: [val_report_dt_best['1'][metric], test_report_dt_best['1'][metric]] for metric in ['precision', 'recall', 'f1-score']}

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['precision', 'recall', 'f1-score']):
    ax[i].bar(['Validation', 'Test'], metrics_values_dt_best[metric], color=['blue', 'green'])
    ax[i].set_title(f'{metric.capitalize()} Comparison after Tuning (Decision Tree)')
    ax[i].set_ylabel(metric.capitalize())
plt.tight_layout()
plt.show()

# Plotting confusion matrix for the best model with annotations in black font
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# For Validation Confusion Matrix
val_conf_matrix_dt_best = confusion_matrix(y_val, y_val_pred_dt_best)
ax[0].imshow(val_conf_matrix_dt_best, cmap='Blues', interpolation='nearest')
ax[0].set_title('Validation Confusion Matrix (Decision Tree - Best Model)')
for i in range(val_conf_matrix_dt_best.shape[0]):
    for j in range(val_conf_matrix_dt_best.shape[1]):
        ax[0].text(j, i, str(val_conf_matrix_dt_best[i, j]), ha='center', va='center', color='black')
ax[0].set_xticks([0, 1])
ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(['Predicted Fail', 'Predicted Pass'])
ax[0].set_yticklabels(['Actual Fail', 'Actual Pass'])

# For Test Confusion Matrix
test_conf_matrix_dt_best = confusion_matrix(y_test, y_test_pred_dt_best)
ax[1].imshow(test_conf_matrix_dt_best, cmap='Blues', interpolation='nearest')
ax[1].set_title('Test Confusion Matrix (Decision Tree - Best Model)')
for i in range(test_conf_matrix_dt_best.shape[0]):
    for j in range(test_conf_matrix_dt_best.shape[1]):
        ax[1].text(j, i, str(test_conf_matrix_dt_best[i, j]), ha='center', va='center', color='black')
ax[1].set_xticks([0, 1])
ax[1].set_yticks([0, 1])
ax[1].set_xticklabels(['Predicted Fail', 'Predicted Pass'])
ax[1].set_yticklabels(['Actual Fail', 'Actual Pass'])

plt.tight_layout()
plt.show()
